#Checkpoint 2 (MovieLens 25M)

Project Scope

Dataset: MovieLens 25M (GroupLens). ~25M ratings, ~62K movies, ~162K users, plus tags and movie metadata.

Possible Course techniques could use:

Graph mining on the user–movie bipartite graph (degrees, projections, centrality/PageRank-style signals).

Similarity-based recommendation baselines (item-item KNN cosine; popularity baseline).

Clustering/segmentation of users or items (based on graph-derived or interaction-derived features).

External technique will learn/use:

Matrix factorization (SVD-style latent factors) as implemented in Surprise.

In [1]:
!pip -q install pandas numpy matplotlib scikit-learn scipy

import zipfile, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = Path("data/ml-25m")
ZIP_PATH = Path("ml-25m.zip")
url = "https://files.grouplens.org/datasets/movielens/ml-25m.zip"

if not DATA_DIR.exists():
    DATA_DIR.parent.mkdir(parents=True, exist_ok=True)
    if not ZIP_PATH.exists():
        urllib.request.urlretrieve(url, ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(DATA_DIR.parent)

ratings = pd.read_csv(DATA_DIR / "ratings.csv")
movies  = pd.read_csv(DATA_DIR / "movies.csv")

ratings["ts"] = pd.to_datetime(ratings["timestamp"], unit="s", errors="coerce")

print("ratings:", ratings.shape, "movies:", movies.shape)
ratings.head()

ratings: (25000095, 5) movies: (62423, 3)


,userId,movieId,rating,timestamp,ts
0,1,296,5.0,1147880044,2006-05-17 15:34:04
1,1,306,3.5,1147868817,2006-05-17 12:26:57
2,1,307,5.0,1147868828,2006-05-17 12:27:08
3,1,665,5.0,1147878820,2006-05-17 15:13:40
4,1,899,3.5,1147868510,2006-05-17 12:21:50


# Research Question Definition (3 RQs)

RQ1 (Course: Graph Mining / Centrality)

Question: How strongly do graph-based centrality signals for movies (built from co-rating structure) correlate with raw popularity, and what does that imply about popularity bias?

* Task type: graph mining / centrality analysis

* Algorithms: build movie–movie co-rating graph (projection), compute centrality (PageRank-style)

* Evaluation: Spearman correlation between centrality and popularity; overlap of top-K movies; qualitative interpretability of top results

RQ2 (Course: Similarity Recommendation + Segmented Evaluation)

Question: How does an item–item cosine similarity recommender perform across different user activity groups (cold/medium/heavy users), compared to a popularity baseline?

* Task type: similarity-based recommendation (top-N)

* Algorithms: popularity baseline; item–item cosine similarity recommender

* Evaluation: Precision@K / Recall@K / NDCG@K, reported separately by activity bins

RQ3 (External + Course comparison: Cold-start + Error Analysis)

Question: For users with very few ratings (or movies with few ratings), which method produces more reasonable recommendations, and what types of errors do we observe (over-popular recommendations, genre mismatch, etc.)?

* Task type: recommendation + error analysis under sparsity/cold-start

* Algorithms:

Course: popularity baseline, item–item cosine similarity

External: matrix factorization (SVD latent factors)

* Evaluation criteria:

Top-N metrics on cold users / rare movies (Precision@K, NDCG@K)

Possible Error :

Over-popular rate (fraction of recs in top popularity percentile)

Genre mismatch (low genre overlap between recs and user’s history/held-out items)

Coverage on rare items (how often recommendations include long-tail movies)

| RQ  | Course technique | External technique | Task type                   | Algorithms                            | Metrics                                                                                   |
| --- | ----------------- | ------------------- | --------------------------- | ------------------------------------- | ----------------------------------------------------------------------------------------- |
| RQ1 | Yes                 | No                   | Graph mining / centrality   | movie projection + PageRank-style     | Spearman corr(centrality, popularity), top-K overlap                                      |
| RQ2 | Yes                 | No                   | Similarity recommendation   | popularity baseline; item–item cosine | Precision@K, Recall@K, NDCG@K (by user bins)                                              |
| RQ3 | Yes (baselines)     | Yes                   | Cold-start + error analysis | popularity, item–item cosine, SVD MF  | Precision@K/NDCG@K on cold/rare + over-popular rate + genre mismatch + long-tail coverage |


# Algorithmic Decisions

Decision: Download MovieLens 25M from the official hosting.

Why: Reproducibility and consistent schema.

Time split (leakage avoidance)

Decision: Use a time-based split (train = earlier ratings, test = later ratings).
Why: Prevents training on future information and better reflects a real recommender setting.

Feasibility constraints for Colab

Decision: Restrict expensive movie–movie computations to TOP_N most-rated movies.
Why: Full movie–movie similarity over all movies is O(M²) and too heavy; TOP_N is a controlled compromise.

Decision: Cap evaluation to a fixed number of users (e.g., 5K) for checkpoint feasibility runs.
Why: Keeps runtime stable while still validating meaningful behavior.

Cold-start definitions (RQ3)

Decision: Define cold users using activity quantiles (e.g., bottom 50% or bottom 10%) and rare movies using popularity thresholds (e.g., bottom percentile or < N ratings in train).
Why: Makes “few ratings” and “rare movies” data-driven rather than arbitrary.

Error analysis choices (RQ3)

Decision: Diagnose recommendation errors with three interpretable signals:

Over-popular rate (are recs dominated by head items?)

Genre mismatch (do rec genres match the user’s preferences or held-out items?)

Long-tail coverage (do we recommend any rare movies?)
Why: These error types directly connect to the EDA issues (heavy tail + sparsity) and are easy to communicate.

External method choice

Decision: Use matrix factorization (SVD latent factors) as the external method.
Why: It is a standard approach for sparse user–item data and is learnable/implementable using an existing library.

In [2]:
n_users = ratings["userId"].nunique()
n_movies = ratings["movieId"].nunique()
n_ratings = len(ratings)
density = n_ratings / (n_users * n_movies)

user_counts = ratings.groupby("userId").size()
movie_counts = ratings.groupby("movieId").size()

print(f"Users={n_users:,} Movies={n_movies:,} Ratings={n_ratings:,} Density≈{density:.6%}")
print("\nUser activity quantiles:")
print(user_counts.quantile([0.1, 0.5, 0.9, 0.99]).astype(int))
print("\nMovie popularity quantiles:")
print(movie_counts.quantile([0.1, 0.5, 0.9, 0.99]).astype(int))

Users=162,541 Movies=59,047 Ratings=25,000,095 Density≈0.260484%

User activity quantiles:
0.10      24
0.50      71
0.90     353
0.99    1228
dtype: int64

Movie popularity quantiles:
0.10       1
0.50       6
0.90     413
0.99    9941
dtype: int64


In [3]:
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity

ratings_sorted = ratings.sort_values("ts")
cut = int(len(ratings_sorted) * 0.8)
train = ratings_sorted.iloc[:cut]
test  = ratings_sorted.iloc[cut:]

TOP_N = 5000
top_movies = train["movieId"].value_counts().head(TOP_N).index

train_sub = train[train["movieId"].isin(top_movies)]
test_sub  = test[test["movieId"].isin(top_movies)]

# Define cold users based on train activity (data-driven)
train_user_counts = train_sub.groupby("userId").size()
q50 = int(train_user_counts.quantile(0.5))
q10 = int(train_user_counts.quantile(0.1))

cold_users_50 = set(train_user_counts[train_user_counts <= q50].index)
cold_users_10 = set(train_user_counts[train_user_counts <= q10].index)

# Define rare movies based on train popularity (within TOP_N space)
train_movie_counts = train_sub.groupby("movieId").size()
rare_cut = int(train_movie_counts.quantile(0.2))  # bottom 20% by count within subset
rare_movies = set(train_movie_counts[train_movie_counts <= rare_cut].index)

print("TOP_N movies:", len(top_movies))
print("Cold users (bottom 50%):", len(cold_users_50), "threshold <=", q50)
print("Cold users (bottom 10%):", len(cold_users_10), "threshold <=", q10)
print("Rare movies (bottom 20% within TOP_N):", len(rare_movies), "threshold <=", rare_cut)

TOP_N movies: 5000
Cold users (bottom 50%): 69205 threshold <= 66
Cold users (bottom 10%): 14258 threshold <= 23
Rare movies (bottom 20% within TOP_N): 1001 threshold <= 666


In [4]:
user_ids = train_sub["userId"].unique()
movie_ids = top_movies.tolist()

u2i = {u:i for i,u in enumerate(user_ids)}
m2i = {m:i for i,m in enumerate(movie_ids)}

R = csr_matrix(
    (train_sub["rating"].astype(float),
     (train_sub["userId"].map(u2i), train_sub["movieId"].map(m2i))),
    shape=(len(user_ids), len(movie_ids))
)

S = cosine_similarity(R.T)
print("Similarity matrix shape:", S.shape)

Similarity matrix shape: (5000, 5000)


In [5]:
# Popularity baseline
pop_ranked = train_sub["movieId"].value_counts().index.tolist()

hist = train_sub.groupby("userId")["movieId"].apply(list).to_dict()
truth = test_sub.groupby("userId")["movieId"].apply(list).to_dict()

sub_index = {m:i for i,m in enumerate(movie_ids)}
inv_index = {i:m for m,i in sub_index.items()}

def recommend_popularity(user_id, k=10):
    seen = set(hist.get(user_id, []))
    return [m for m in pop_ranked if m not in seen][:k]

def recommend_item_item(user_id, k=10):
    seen = hist.get(user_id, [])
    seen_idx = [sub_index[m] for m in seen if m in sub_index]
    if not seen_idx:
        return []
    scores = S[seen_idx].max(axis=0)
    for idx in seen_idx:
        scores[idx] = -np.inf
    topk = np.argsort(-scores)[:k]
    return [inv_index[i] for i in topk if np.isfinite(scores[i])]

def precision_at_k(recs, truth_items, k=10):
    recs = recs[:k]
    if len(recs) == 0:
        return 0.0
    return len(set(recs) & set(truth_items)) / k

In [6]:
# Build movieId -> genres set
movies_gen = movies.copy()
movies_gen["genres_set"] = movies_gen["genres"].fillna("").apply(lambda g: set(g.split("|")) if g != "(no genres listed)" else set())
mid2genres = dict(zip(movies_gen["movieId"], movies_gen["genres_set"]))

# Popularity percentile within TOP_N train subset
pop_counts = train_sub["movieId"].value_counts()
pop_rank = pop_counts.rank(pct=True, ascending=False)  # 0..1 where 1 means most popular (because rank pct)
# Convert to "popularity percentile" where higher = more popular
mid2pop_pct = pop_rank.to_dict()

def genre_overlap(a_set, b_set):
    if not a_set and not b_set:
        return 1.0
    if not a_set or not b_set:
        return 0.0
    return len(a_set & b_set) / len(a_set | b_set)

In [7]:
def evaluate_user_recs(user_id, recs, truth_items):
    """
    Returns interpretable error diagnostics:
    - over_popular_rate: fraction of recs in top 10% popularity (within subset)
    - avg_genre_match_hist: avg genre overlap between recs and user's train history
    - avg_genre_match_truth: avg genre overlap between recs and heldout truth items
    - long_tail_coverage: fraction of recs that are in rare_movies
    - precision@10
    """
    seen = hist.get(user_id, [])
    seen_genres = set().union(*[mid2genres.get(m, set()) for m in seen]) if seen else set()
    truth_genres = set().union(*[mid2genres.get(m, set()) for m in truth_items]) if truth_items else set()

    rec_genre_matches_hist = []
    rec_genre_matches_truth = []
    over_pop = 0
    long_tail = 0

    for m in recs:
        g = mid2genres.get(m, set())
        rec_genre_matches_hist.append(genre_overlap(g, seen_genres))
        rec_genre_matches_truth.append(genre_overlap(g, truth_genres))

        if mid2pop_pct.get(m, 0.0) >= 0.90:  # top 10% popular
            over_pop += 1
        if m in rare_movies:
            long_tail += 1

    return {
        "precision@10": precision_at_k(recs, truth_items, 10),
        "over_popular_rate": over_pop / max(len(recs), 1),
        "avg_genre_match_hist": float(np.mean(rec_genre_matches_hist)) if rec_genre_matches_hist else 0.0,
        "avg_genre_match_truth": float(np.mean(rec_genre_matches_truth)) if rec_genre_matches_truth else 0.0,
        "long_tail_coverage": long_tail / max(len(recs), 1)
    }

# Evaluate on cold users (choose bottom 10% for strict cold-start)
users_eval = [u for u in truth.keys() if u in hist and u in cold_users_10][:3000]

rows = []
for u in users_eval:
    t = truth[u]
    rec_pop = recommend_popularity(u, 10)
    rec_knn = recommend_item_item(u, 10)

    d_pop = evaluate_user_recs(u, rec_pop, t)
    d_knn = evaluate_user_recs(u, rec_knn, t)

    rows.append({
        "userId": u,
        "model": "popularity",
        **d_pop
    })
    rows.append({
        "userId": u,
        "model": "item_item",
        **d_knn
    })

diag_df = pd.DataFrame(rows)
diag_df.groupby("model")[["precision@10","over_popular_rate","avg_genre_match_hist","avg_genre_match_truth","long_tail_coverage"]].mean()

,precision@10,over_popular_rate,avg_genre_match_hist,avg_genre_match_truth,long_tail_coverage
model,,,,,
item_item,0.201609,0.001149,0.246072,0.203651,0.001379
popularity,0.207816,0.000000,0.218266,0.200990,0.000000


# Motivation

EDA suggests sparsity and a long-tail of users/movies with few interactions. That motivates asking which method behaves most “reasonably” under cold-start conditions, and what mistakes are common.


Similarity methods depend on overlap/co-ratings, which may fail for cold users and rare movies. The external MF method can learn latent structure that may help under sparsity.

Feasibility evidence (from initial runs)

Similarity computations are feasible with TOP_N movies.

Cold-user evaluation is feasible with user caps (e.g., 3K).

MF feasibility is demonstrated using Surprise SVD trained on a sample.

Risks and mitigation

TOP_N restriction biases toward popular movies -> we’ll repeat experiments with multiple TOP_N values later.

MF hyperparameters may matter -> we start with stable defaults and plan tuning later.

Genre mismatch is an approximation using metadata genres -> still interpretable for error analysis, but we’ll note its limits.

# Method & Metric Plan

RQ1: movie–movie projection and centrality; compare centrality vs popularity.

RQ2: popularity baseline vs item–item cosine similarity; evaluate across user bins.

RQ3: compare popularity vs item–item vs SVD specifically on cold users and rare movies, plus error breakdown.

Metrics

Ranking quality: Precision@K, NDCG@K

Cold-start reasonableness diagnostics (RQ3):

Over-popular rate (how many recommendations are in top popularity percentiles)

Genre mismatch (genre overlap between recommended and user history / held-out)

Long-tail coverage (fraction of recommendations that are rare movies)

# Baselines

Popularity baseline is a must-have baseline.

Item–item cosine is the core course baseline.

SVD is the external technique baseline.

Notes for the RQ3

To strengthen RQ3’s “types of errors” claim, have tried produces measurable proxies:

“over-popular” -> high popularity percentile fraction

“genre mismatch” -> low genre overlap score

“long-tail coverage” -> fraction of rare movies recommended

On my honor, I declare the following resources:
1. Collaborators:
- none

2. Web Sources:
- https://grouplens.org/datasets/movielens/25m/


3. AI Tools:
- ChatGPT: I prompted to get some deeper dive in the dataset and some research ideas of the datasets I could try, and get help building the EDA and finding interesting fetures for the Movie rating dataset.

4. Citations:
- F. Maxwell Harper and Joseph A. Konstan. 2015. The MovieLens Datasets: History and Context. ACM Transactions on Interactive Intelligent Systems (TiiS) 5, 4: 19:1–19:19. https://doi.org/10.1145/2827872
